In [1]:
from cvtypecorr.data.rxn import FormationReaction, ThermoReaction, RedoxReaction, AcidDeprotReaction
from cvtypecorr.fit.rigid import Collection

# We can gather a collection of dft correction energies by first defining a Collection

corrs = Collection(
    corrections = {}, # pre-defined corrections can be added within initialization
)
corrs.corrections["H2O"] = 0.0 # These corrections are just a standard dict[str, float] without any special properties, so they can be added to as needed.
corrs.corrections.update({"H3O+": 0.0}) # Or updated with a batch of corrections at once
ex_corrections = corrs.corrections # Can be fetched as expected
print(ex_corrections)


{'H2O': 0.0, 'H3O+': 0.0}


In [2]:
from cvtypecorr.data.pka import data_dict

# Let's apply corrections for NO3H -> NO3- -> NO2 -> NO
# We can start by defining to NO3H -> NO3- reaction, which will inform us on corrections for NO3-.

acid = "NO3H"
base = "NO3-"
acid_pka = data_dict[acid]

no3h_to_no3_acid_deprot = AcidDeprotReaction(
    "NO3H -> NO3- + H+",
    acid, base, acid_pka,
    ref_acid_base_pair_and_pKa=("H3O+", "H2O", 0.0), T=298.15, 
)

In [3]:
# Rigid corrections can only update one specie at a time, so since we are missing both the acid and base,
# this will raise an error.
try:
    corrs.apply_acid_correction(no3h_to_no3_acid_deprot, -4.0)
except ValueError as e:
    print(e)
    pass

Rigid correction can only apply fit correction at a time (currently missing [('NO3H', 1)] and [('NO3-', 1)])


In [4]:
# We can fix this by grounding one of the two to have a predefined correction.
# Since ionic species have much higher DFT error, we can set the correction for NO3H to 0, allowing us
# to obtain a correction for NO3-.
corrs.corrections["NO3H"] = 0.0
corrs.apply_acid_correction(no3h_to_no3_acid_deprot, -4.0, overwrite="NO3-") # Putting NO3- in overwrite will remove NO3- from corrections if already present from a previous cell run

# Now we have a correction value to add onto the DFT energy for NO3- to enforce agreement with the experimental pKa. 
# Since -4 is less than experimental pKa, our DFT NO3H acid strength is too strong, thus either NO3H is to unstable or NO3- is too stable. 
# Since we set the correction for NO3H to 0, we should see a positive correction for NO3-, effectively destabilizing NO3-.
print("E_err for NO3-:", corrs.corrections["NO3-"])

E_err for NO3-: 0.15973024414891232


In [5]:
# Next we can skip ahead and get a correction for NO with the known redox potential of NO3- to NO

no3_to_no_reduction = RedoxReaction(
    "NO3 + 3 e- -> NO + 2H2O",
    [("NO3-", 1), ("e-", 3), ("H3O+", 4)], # Must give electrons as a reactant for reduction, or as products for oxidation.
    [("NO", 1), ("H2O", 6)],
    0.96, # This is the standard reduction potential in V vs SHE.
    v0 = 4.66, # For non-CANDLE reactions, we can change this to another calibrated μ_SHE.
)
corrs.apply_redox_correction(no3_to_no_reduction, 0.8, overwrite="NO") # Note this DFT-derived reduction potential should already include the corrections found at this point.

# Since our DFT reduction potential is more negative than the experimental reduction potential, our DFT NO3- to NO reduction is too uphill, 
# thus either NO3- is too stable or NO is too unstable. Since a correction for NO3- is already defined, 
# we should see a negative correction for NO, effectively stabilizing NO.
print("E_err for NO:", corrs.corrections["NO"])

E_err for NO: -0.4800000000000004


In [6]:
# Next we can reverse to get NO3H from NO

no3_to_no_reduction = RedoxReaction(
    "NO2H + e- + H3O+ -> NO + 2H2O",
    [("NO2H", 1), ("e-", 1), ("H3O+", 1)], # Must give electrons as a reactant for reduction, or as products for oxidation.
    [("NO", 1), ("H2O", 2)],
    0.996, # This is the standard reduction potential in V vs SHE.
    v0 = 4.66, # For non-CANDLE reactions, we can change this to another calibrated μ_SHE.
)
corrs.apply_redox_correction(no3_to_no_reduction, 1.1, overwrite="NO2H") 
print("E_err for NO2H:", corrs.corrections["NO2H"])

E_err for NO2H: -0.1039999999999992


In [9]:
# And finally get NO2- from NO2H

acid = "NO2H"
base = "NO2-"
acid_pka = data_dict[acid]

no3h_to_no3_acid_deprot = AcidDeprotReaction(
    "NO2H -> NO2- + H+",
    acid, base, acid_pka,
    ref_acid_base_pair_and_pKa=("H3O+", "H2O", 0.0), T=298.15, 
)

corrs.apply_acid_correction(no3h_to_no3_acid_deprot, 4.0, overwrite="NO2-")
print("E_err for NO2-:", corrs.corrections["NO2-"])

E_err for NO2-: -0.04200313827619545
